#DATA CLEANING & PREPROCESSING FOR AMAZON REVIEWS




In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MinMaxScaler

In [3]:
# Load Dataset

df = pd.read_csv("amazon_reviews_2000_15cols.csv")

print("\n Available Columns in Your Dataset:")
print(df.columns)


 Available Columns in Your Dataset:
Index(['review_id', 'user_id', 'product_id', 'category', 'subcategory',
       'rating', 'review_text', 'timestamp', 'price', 'brand', 'helpful_votes',
       'verified_purchase', 'sentiment', 'country', 'product_title'],
      dtype='object')


In [5]:
# Step 1: Detect review and summary columns

text_col = None
summary_col = None
rating_col = None

In [9]:
# Possible names
text_options = ["reviewText", "review_text", "text", "review", "content", "body", "review_body"]
summary_options = ["summary", "title", "headline", "review_summary", "reviewTitle"]
rating_options = ["rating", "stars", "overall", "rating_value", "star_rating"]

# Auto-detect
for col in df.columns:
    col_lower = col.lower()
    if col_lower in [x.lower() for x in text_options]:
        text_col = col
    if col_lower in [x.lower() for x in summary_options]:
        summary_col = col
    if col_lower in [x.lower() for x in rating_options]:
        rating_col = col




In [11]:
# If not found → create dummy empty columns
if text_col is None:
    print("\n Review text column not found — creating one.")
    df["review_text_auto"] = ""
    text_col = "review_text_auto"

if summary_col is None:
    print("\n Summary column not found — creating one.")
    df["summary_auto"] = ""
    summary_col = "summary_auto"

if rating_col is None:
    print("\n Rating column not found — creating one with default value 3.")
    df["rating_auto"] = 3
    rating_col = "rating_auto"

print("\n Using review column:", text_col)
print(" Using summary column:", summary_col)
print(" Using rating column:", rating_col)


 Summary column not found — creating one.

 Using review column: review_text
 Using summary column: summary_auto
 Using rating column: rating


In [15]:
# Step 2: HANDLE MISSING VALUe
# Fill missing text
df[text_col] = df[text_col].fillna("")
df[summary_col] = df[summary_col].fillna("")



In [17]:
# Fill missing numerical values 
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [21]:
# Step 3: NORMALIZE RATINGS (0-1 scale)

scaler = MinMaxScaler()
df["rating_normalized"] = scaler.fit_transform(df[[rating_col]])

In [25]:
# Step 4: Convert date column to datetime

date_options = ["reviewTime", "date", "review_date"]

for col in df.columns:
    if col.lower() in [x.lower() for x in date_options]:
        df[col] = pd.to_datetime(df[col], errors="coerce")

In [27]:
# Step 5: Clean Text Function

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_review"] = df[text_col].apply(clean_text)
df["clean_summary"] = df[summary_col].apply(clean_text)


In [29]:
# Step 6: Remove Duplicate Reviews

df = df.drop_duplicates(subset=[text_col, summary_col], keep="first")


In [31]:
# Step 7: Final Output
# -----------------------------------------
print("\n Cleaning & preprocessing completed!")
print(" Final Shape:", df.shape)


 Cleaning & preprocessing completed!
 Final Shape: (8, 19)


In [37]:
# Step 8: Save Cleaned File

df.to_csv("cleaned_amazon_reviews.csv", index=False)
print("\n File saved as: cleaned_amazon_reviews.csv")


 File saved as: cleaned_amazon_reviews.csv
